# Pandas from First Principles
## Notebook 1: Foundations & the Series

---

**Who this is for:** Python developers who know functions, OOP, list comprehensions, and generators — but have never seriously used Pandas.

**What you will build by the end:** A mental model of *why* Pandas exists, *what* it is made of, and *how* to think in its idioms — not just a list of methods.

---

### Table of Contents

1. Why Pandas Exists
2. Pandas under the Hood — A Mental Model
3. The Series — Pandas' Fundamental Building Block
4. Creating a Series
5. The Index — Pandas' Secret Weapon
6. Selecting Data from a Series
7. Vectorized Operations
8. Handling Missing Data in a Series
9. Useful Series Methods
10. End-of-Notebook Summary, Cheat Sheet & Exercises

---

# 1. Why Pandas Exists

## What is the problem?

Before you learn a tool, understand what pain it removes.

Imagine you have a CSV file with 100,000 rows of sales data. You need to:

- Read it into memory
- Filter rows where `revenue > 50000`
- Group by `region` and compute average revenue
- Sort by that average, descending
- Write the result to a new CSV

**Without Pandas**, you would reach for:
- `csv.reader` or `open()` to read the file
- Python lists of dicts to store rows
- Manual `for` loops for filtering
- Nested dicts for grouping
- Lots of boilerplate just to answer a simple question

Let's feel this pain first.

In [ ]:
# Simulating the 'raw Python' approach to a data question
sales_data = [
    {"region": "North", "revenue": 82000, "product": "Laptop"},
    {"region": "South", "revenue": 45000, "product": "Phone"},
    {"region": "North", "revenue": 61000, "product": "Tablet"},
    {"region": "South", "revenue": 55000, "product": "Laptop"},
    {"region": "East",  "revenue": 73000, "product": "Phone"},
    {"region": "East",  "revenue": 41000, "product": "Tablet"},
]

# Step 1: Filter rows where revenue > 50000
filtered = [row for row in sales_data if row["revenue"] > 50000]

# Step 2: Group by region and sum revenue
region_totals = {}
region_counts = {}
for row in filtered:
    region = row["region"]
    region_totals[region] = region_totals.get(region, 0) + row["revenue"]
    region_counts[region] = region_counts.get(region, 0) + 1

# Step 3: Compute average per region
region_avg = {
    region: region_totals[region] / region_counts[region]
    for region in region_totals
}

# Step 4: Sort descending by average
sorted_regions = sorted(region_avg.items(), key=lambda x: x[1], reverse=True)

print("Region | Avg Revenue (filtered)")
print("-" * 35)
for region, avg in sorted_regions:
    print(f"{region:<8} | {avg:,.0f}")

That works. But notice what happened:

- You wrote ~15 lines of logic just to answer one question.
- It is fragile: what if a key is missing in one dict?
- It is slow: Python `for` loops are interpreted one element at a time.
- It does not scale: at 10 million rows, this becomes painfully slow.

Now watch the same task with Pandas.

In [ ]:
import pandas as pd

# Same data, Pandas version
df = pd.DataFrame(sales_data)

result = (
    df[df["revenue"] > 50000]
    .groupby("region")["revenue"]
    .mean()
    .sort_values(ascending=False)
)

print(result)

**5 lines. Same result.**

And at 10 million rows, Pandas would still be fast because it delegates heavy computation to NumPy, which runs compiled C/Fortran code — not Python loops.

## So what exactly is Pandas?

Pandas is a Python library built on top of **NumPy** that provides:

- Two core data structures: **Series** (1D) and **DataFrame** (2D)
- A rich vocabulary for **selecting, filtering, grouping, reshaping, and aggregating** tabular data
- Native support for **missing values, time series, and labeled data**
- Easy I/O: read from and write to CSV, Excel, SQL, JSON, Parquet, and more

The name comes from **Pan**el **Da**ta — a term from econometrics for multi-dimensional structured data.

## When to use Pandas

| Situation | Use Pandas? |
|---|---|
| Tabular data (rows and columns) | Yes |
| CSV / Excel / SQL data | Yes |
| Data cleaning and transformation | Yes |
| Exploratory data analysis | Yes |
| Data fits in RAM (< a few GB) | Yes |
| Data does not fit in RAM | No — use Dask, Polars, or Spark |
| Pure number crunching (matrices) | No — use NumPy directly |
| Real-time streaming data | No — use Kafka / Flink |

## When NOT to use Pandas

- Your data is larger than ~8 GB — Pandas loads everything into RAM.
- You need maximum performance on pure numeric arrays — NumPy is faster with less overhead.
- You need concurrent, multi-threaded computation — look at Polars instead.
- Your data is entirely unstructured text — NLP pipelines have better tools.

**Interview Note:** A common question is "What is the difference between Pandas and NumPy?" Pandas is built *on* NumPy. NumPy gives you fast array math. Pandas adds labeled axes (the Index), heterogeneous column types, missing value support, and a rich data manipulation API. You would not typically choose one over the other — you often use both.

---

# 2. Pandas under the Hood — A Mental Model

## How Pandas stores data internally

This is not deep theory — it is the minimum you need to avoid confusion later.

Under the hood, a Pandas **Series** is essentially:

```
Series
  ├── data   → a NumPy array  (the actual values)
  └── index  → a NumPy array  (the labels for those values)
```

A Pandas **DataFrame** is:

```
DataFrame
  ├── columns  → a collection of named Series objects (each column is a Series)
  └── index    → shared row labels across all columns
```

This architecture means:
- Operations on Series/DataFrames often delegate to NumPy, which is written in C — so they are fast.
- The **Index** is a first-class citizen. It is not just row numbers — it can be strings, dates, or anything.
- Every column in a DataFrame has a single dtype. Mixed-type columns fall back to `object` dtype (essentially Python objects), which is slow.

Keep this picture in your head as you proceed. It will explain a lot of behavior that otherwise seems arbitrary.

In [ ]:
import numpy as np
import pandas as pd

# Verifying the mental model
temperatures = pd.Series([22.5, 18.0, 25.3, 30.1, 19.8])

print("Type:                ", type(temperatures))
print("Values (NumPy array):", temperatures.values)
print("Type of .values:     ", type(temperatures.values))
print("Index:               ", temperatures.index)
print("dtype:               ", temperatures.dtype)

Notice `.values` returns a plain NumPy array. The Index here is `RangeIndex(start=0, stop=5, step=1)` — Pandas' efficient default when you do not specify labels. It behaves like `range(5)` and barely uses any memory.

Now let's go deep on the **Series**.

---

# 3. The Series — Pandas' Fundamental Building Block

## What is a Series?

A Series is a **one-dimensional, labeled array** that can hold any single data type: integers, floats, strings, booleans, Python objects, or datetime values.

Think of it as a Python list with two superpowers:
1. Every element has a **label** (the index), not just a position.
2. Operations on it run at near-C speed via NumPy.

## Real-world analogy

Imagine a single column in a spreadsheet. If you look at a column called "Temperature" in Excel:

```
Row | Temperature
----|------------
  1 |   22.5
  2 |   18.0
  3 |   25.3
```

That is essentially a Series. The "Row" column is the **index**. The "Temperature" column is the **values**.

But the index does not have to be numbers. It could be city names, dates, or product IDs. That labeling is what makes Series (and DataFrames) qualitatively different from raw arrays.

## Why not just use a Python list?

| Feature | Python list | Pandas Series |
|---|---|---|
| Labels (non-integer index) | No | Yes |
| Fast vectorized math | No | Yes |
| Missing value handling | Manual | Built-in (`NaN`) |
| Alignment on operations | No | Yes |
| Rich built-in statistics | No | Yes |
| Data type enforcement | No | Yes (single dtype) |

---

# 4. Creating a Series

## Syntax

```python
pd.Series(data, index=None, dtype=None, name=None)
```

## Parameters

| Parameter | What it does | Default |
|---|---|---|
| `data` | The actual values (list, dict, scalar, ndarray) | Required |
| `index` | Labels for each element. Must match length of `data` | `RangeIndex` (0, 1, 2...) |
| `dtype` | Force a specific data type (e.g., `float32`, `str`) | Inferred |
| `name` | A label for the Series itself | `None` |

## Return Value

Returns a `pandas.Series` object. The original data (if it was a list or NumPy array) is **not mutated**.

---

## 4.1 From a list

In [ ]:
import pandas as pd

# Basic: Series from a list. Index is auto-assigned (0, 1, 2, ...)
daily_temps = pd.Series([22.5, 18.0, 25.3, 30.1, 19.8])
print(daily_temps)
print("\nIndex:", daily_temps.index)
print("dtype:", daily_temps.dtype)

In [ ]:
# Practical: Give it meaningful labels using the index parameter
days  = ["Mon", "Tue", "Wed", "Thu", "Fri"]
temps = [22.5, 18.0, 25.3, 30.1, 19.8]

weekly_temps = pd.Series(data=temps, index=days, name="temperature_celsius")
print(weekly_temps)

In [ ]:
# Now you can access by label — much more readable than weekly_temps[0]
print("Wednesday temperature:", weekly_temps["Wed"])
print("Series name:          ", weekly_temps.name)

## 4.2 From a dictionary

This is probably the most intuitive way to create a Series when you already have labeled data. The dictionary **keys become the index** and the values become the data.

In [ ]:
# Creating a Series from a dictionary
stock_prices = {
    "AAPL":  189.30,
    "GOOGL": 141.80,
    "MSFT":  415.50,
    "AMZN":  185.70,
}

prices = pd.Series(stock_prices, name="closing_price_usd")
print(prices)

In [ ]:
# You can filter which keys to include by passing an index.
# Keys not in the dict become NaN (missing).
partial_prices = pd.Series(
    stock_prices,
    index=["AAPL", "GOOGL", "TSLA"],  # TSLA is not in the dict
    name="closing_price_usd"
)
print(partial_prices)
print("\nNotice: TSLA has NaN because it was not in the source dict")

**Key insight:** When you create a Series from a dict and also provide an explicit `index`, Pandas **aligns by label**. Labels in `index` that are not found in the dict silently become `NaN`. This alignment behavior is a core Pandas concept — you will see it everywhere.

## 4.3 From a scalar (broadcasting)

In [ ]:
# A single scalar value broadcast across all index labels.
# Useful for initializing a Series with a default value.
default_scores = pd.Series(data=0, index=["Alice", "Bob", "Carol", "Dave"])
print(default_scores)
print("\nAll values start at 0 — useful for initializing accumulators")

## 4.4 Controlling dtype

In [ ]:
# By default, Pandas infers the dtype
inferred = pd.Series([1, 2, 3, 4])
print("Inferred dtype:", inferred.dtype)   # int64

# Force a specific dtype for memory efficiency or correctness
as_float = pd.Series([1, 2, 3, 4], dtype="float32")
print("Forced dtype:  ", as_float.dtype)

as_bool = pd.Series([1, 0, 1, 1], dtype="bool")
print("As boolean:    ", as_bool.values)

## Common dtypes you will encounter

| Pandas dtype | Python equivalent | When you see it |
|---|---|---|
| `int64` | `int` | Integer columns |
| `float64` | `float` | Float columns, or int columns with missing values |
| `bool` | `bool` | Boolean columns |
| `object` | `str` (usually) | String columns, mixed-type columns |
| `datetime64[ns]` | `datetime` | Date/time columns |
| `category` | — | Low-cardinality string columns (memory efficient) |

**Performance note:** `object` dtype stores Python objects, which means each element is a full Python object in memory. For string data, this is 5–10x more memory-intensive than numeric dtypes. For large string columns, `category` dtype is much more efficient.

**Common mistake:** Storing numbers as `object` dtype because one row had a string like `"N/A"` mixed in. Always inspect dtypes after loading data.

In [ ]:
import numpy as np

# Common mistake: mixed data causes object dtype
bad_series = pd.Series([1, 2, "N/A", 4])  # One string ruins it
print("dtype with mixed data:", bad_series.dtype)  # object
print(bad_series)

print("\n--- Correct approach: represent missing as None or np.nan ---")
good_series = pd.Series([1, 2, np.nan, 4])  # NaN is a float sentinel
print("dtype with NaN:       ", good_series.dtype)  # float64
print(good_series)

---

# 5. The Index — Pandas' Secret Weapon

## What is the Index?

The Index is a separate data structure attached to every Series (and DataFrame). It stores the **labels** for the data.

Most beginners think of the index as just "row numbers." That is a dangerous mental model. The index is a **first-class object** with its own type, its own methods, and its own rules.

## Why it matters

The index enables **label-based alignment**. When you add two Series together, Pandas does not add element-by-element like a Python list. It adds by **matching labels**. This prevents a whole class of subtle bugs.

## Analogy

Think of a Series as a Python dictionary. You never add two dicts by position — you work with their keys. The index is Pandas' version of dictionary keys. That is why creating a Series from a dict feels so natural.

In [ ]:
# Alignment in action — the most important thing about the Index

# Q1 sales by region
q1_sales = pd.Series({"North": 120_000, "South": 95_000, "East": 143_000})

# Q2 sales — different order, and 'West' region added
q2_sales = pd.Series({"South": 101_000, "North": 131_000, "West": 88_000, "East": 155_000})

# Pandas aligns by label automatically — no manual sorting needed
total_sales = q1_sales + q2_sales

print("Q1 sales:")
print(q1_sales)
print("\nQ2 sales:")
print(q2_sales)
print("\nTotal (auto-aligned by label):")
print(total_sales)
print("\nNote: 'West' is NaN because it only existed in Q2")

This alignment behavior is **fundamentally different from NumPy arrays**, which add element-by-element based on position. With NumPy, if the arrays have different lengths you get an error. If they are the same length but data is in a different order, you silently get wrong results.

Pandas' label alignment catches those bugs for you.

## Index types

In [ ]:
# RangeIndex: the default, memory-efficient
s1 = pd.Series([10, 20, 30])
print("RangeIndex:    ", s1.index)

# Index: generic index (strings, integers, etc.)
s2 = pd.Series([10, 20, 30], index=["a", "b", "c"])
print("String Index:  ", s2.index)

# DatetimeIndex: for time series data
dates = pd.date_range(start="2024-01-01", periods=3, freq="D")
s3 = pd.Series([100, 200, 150], index=dates)
print("DatetimeIndex: ", s3.index)

In [ ]:
# Index properties you will use regularly
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])

print("Index:        ", s.index.tolist())
print("Values:       ", s.values)
print("Shape:        ", s.shape)          # (4,) — a tuple, like NumPy
print("Length:       ", len(s))
print("Dtype:        ", s.dtype)
print("Is unique?:   ", s.index.is_unique)
print("Is monotonic?:", s.index.is_monotonic_increasing)

## Can an index have duplicate values?

Yes — and this is a source of bugs for many beginners.

In [ ]:
# Duplicate index labels are allowed
s = pd.Series([10, 20, 30], index=["a", "a", "b"])
print(s)

# But accessing a duplicate label returns MULTIPLE values (a Series, not a scalar)
print("\ns['a'] returns:")
print(s["a"])   # Returns a Series with two rows, not a scalar!
print("\nType:", type(s["a"]))

**Interview insight:** If your index has duplicates, label-based access becomes ambiguous. This is a common source of bugs. Always check `s.index.is_unique` if you plan to rely on label-based access. Use `.reset_index()` to restore a clean integer index.

---

# 6. Selecting Data from a Series

## The two ways to select: `.loc` vs `.iloc`

This is where beginners often get confused. Pandas has two primary selection tools:

| Accessor | Stands for | Selects by |
|---|---|---|
| `.loc[]` | **Loc**ation (label) | Index **labels** |
| `.iloc[]` | **I**nteger **loc**ation | Integer **positions** (0, 1, 2, ...) |

## Rule of thumb

Use `.loc` when you want to say **"give me the row labeled X"**.
Use `.iloc` when you want to say **"give me the N-th row"**.

## Why not just use `[]` directly?

`series[key]` is ambiguous and behavior depends on the index type. When the index is integers, `series[2]` is label-based (selects the row labeled `2`), which may or may not be the third row. This creates silent bugs. Always prefer `.loc` or `.iloc` for clarity.

In [ ]:
# Setup: a Series with meaningful string labels
city_populations = pd.Series(
    data=[1_400_000, 2_100_000, 980_000, 3_500_000, 760_000],
    index=["Delhi", "Mumbai", "Pune", "Kolkata", "Jaipur"],
    name="population"
)
print(city_populations)

## 6.1 `.loc[]` — Label-based selection

In [ ]:
# Single label — returns a scalar
print("Mumbai population:", city_populations.loc["Mumbai"])

# List of labels — returns a Series
print("\nMultiple cities:")
print(city_populations.loc[["Delhi", "Jaipur"]])

# Slice with labels — INCLUSIVE on both ends (unlike Python slices!)
print("\nSlice from Delhi to Kolkata (inclusive):")
print(city_populations.loc["Delhi":"Kolkata"])

**Critical difference from Python slicing:** Python's `list[a:b]` excludes `b`. But `.loc["a":"b"]` **includes** `b`. This is intentional — with string labels, you cannot always know what would come "one after" a label.

## 6.2 `.iloc[]` — Integer position-based selection

In [ ]:
# Single position — returns a scalar
print("First city (position 0):  ", city_populations.iloc[0])

# Last element — negative indexing works like Python lists
print("Last city (position -1):  ", city_populations.iloc[-1])

# List of positions — returns a Series
print("\nPositions 0 and 3:")
print(city_populations.iloc[[0, 3]])

# Slice with positions — EXCLUSIVE on the right end (like Python)
print("\nFirst three cities (positions 0, 1, 2):")
print(city_populations.iloc[0:3])

## 6.3 Boolean indexing — the most powerful selector

In [ ]:
# A boolean mask is a Series of True/False values with the same index
large_cities_mask = city_populations > 1_500_000
print("Mask (True where population > 1.5M):")
print(large_cities_mask)

# Apply the mask — only True rows are returned
print("\nCities with population > 1.5 million:")
print(city_populations[large_cities_mask])

In [ ]:
# Combining conditions with & (and), | (or), ~ (not)
# IMPORTANT: Use parentheses around each condition.
# Python operator precedence will cause errors without them.

medium_cities = city_populations[
    (city_populations > 900_000) & (city_populations < 2_500_000)
]
print("Population between 900K and 2.5M:")
print(medium_cities)

**Common mistake — the Python `and` trap:**

```python
# WRONG — Python's 'and' does not work element-wise on Series
city_populations[(city_populations > 900_000) and (city_populations < 2_500_000)]
# → ValueError: The truth value of a Series is ambiguous

# CORRECT — use & for element-wise AND
city_populations[(city_populations > 900_000) & (city_populations < 2_500_000)]
```

Always use `&`, `|`, `~` for element-wise logical operations on Series.

In [ ]:
# Demonstrating the error so you recognize it
try:
    result = city_populations[(city_populations > 900_000) and (city_populations < 2_500_000)]
except ValueError as e:
    print(f"ValueError: {e}")
    print("\nLesson: Use '&' not 'and' for element-wise AND on Series")

## 6.4 The `.loc` / `.iloc` confusion trap — integer index

In [ ]:
# The most dangerous scenario: a non-default integer index.
# This happens whenever you slice a Series — it keeps original labels.

original = pd.Series([10, 20, 30, 40, 50])
sliced = original[2:]  # This KEEPS the original integer labels!

print("Sliced Series (notice index starts at 2):")
print(sliced)

print("\nsliced.loc[2]  → label 2    → value:", sliced.loc[2])
print("sliced.iloc[0] → position 0 → value:", sliced.iloc[0])

After slicing, the index labels are **2, 3, 4** — not **0, 1, 2**. So `sliced.loc[2]` gives you the value at **label 2**, while `sliced.iloc[0]` gives you the **first element by position**. Being explicit about `.loc` vs `.iloc` protects you from this class of bugs.

**Interview insight:** Interviewers often ask about the difference between `.loc` and `.iloc`. Key answer: `.loc` is label-based and inclusive on both ends of slices. `.iloc` is position-based and exclusive on the right end (like Python). When you slice a Series and the index is integers, these two can return different results.

---

# 7. Vectorized Operations

## What are vectorized operations?

A vectorized operation applies a computation to an **entire array at once** using compiled C code, rather than looping through each element in Python.

## Why does it matter?

Python `for` loops have overhead on every iteration: type checking, object allocation, method dispatch. For large arrays, this adds up to seconds or minutes. NumPy/Pandas vectorized operations bypass all of that and run at C speed.

## Intuition

Imagine you are grading 10,000 test papers. You could grade them one by one (a loop), or you could use a scanning machine that processes all of them simultaneously (vectorized). Same result, wildly different speed.

In [ ]:
import time

# Create a large Series to compare speed
large_series = pd.Series(np.random.randint(1, 1000, size=1_000_000))

# Method 1: Python loop
start = time.time()
loop_result = [x * 2 + 10 for x in large_series]
loop_time = time.time() - start

# Method 2: Pandas vectorized operation
start = time.time()
vectorized_result = large_series * 2 + 10
vectorized_time = time.time() - start

print(f"Python loop:       {loop_time:.4f} seconds")
print(f"Pandas vectorized: {vectorized_time:.4f} seconds")
print(f"Speedup:           {loop_time / vectorized_time:.1f}x faster")

## 7.1 Arithmetic operations

In [ ]:
# Real-world: Convert USD prices to INR
prices_usd = pd.Series(
    [19.99, 49.99, 9.99, 99.99, 5.49],
    index=["book", "course", "app", "software", "sticker"]
)

usd_to_inr = 83.5

# Applied to every element — no loop needed
prices_inr = (prices_usd * usd_to_inr).round(2)

print("Price in USD:")
print(prices_usd)
print("\nPrice in INR:")
print(prices_inr)

In [ ]:
# Operations between two Series auto-align by index label
q1_revenue = pd.Series({"North": 120_000, "South": 95_000, "East": 143_000})
q1_cost    = pd.Series({"South": 70_000,  "North": 80_000,  "East": 95_000})

# Even though the order is different, alignment by label gives correct results
q1_profit = q1_revenue - q1_cost

print("Q1 Profit by Region:")
print(q1_profit)

## 7.2 Applying functions with `.apply()`

## Syntax

```python
series.apply(func, args=(), **kwargs)
```

When a vectorized operation is not enough (e.g., you have a complex custom function), use `.apply()`. It calls your function on each element. It is much slower than true vectorized ops, but far more flexible.

**Important note:** `.apply()` is still a Python-level loop internally. Use it when NumPy vectorization is not possible — not as a default.

In [ ]:
# apply() with a custom function
def classify_temperature(temp_celsius):
    if temp_celsius < 15:
        return "cold"
    elif temp_celsius < 25:
        return "comfortable"
    else:
        return "hot"

weekly_temps = pd.Series(
    [22.5, 18.0, 25.3, 30.1, 12.8],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"]
)

comfort_ratings = weekly_temps.apply(classify_temperature)
print(comfort_ratings)

In [ ]:
# apply() with a lambda — compact, but same speed as a named function
is_hot_apply = weekly_temps.apply(lambda t: t > 25)

# Better: vectorized comparison avoids Python-level loop entirely
is_hot_vectorized = weekly_temps > 25

print("apply() result:     ", is_hot_apply.tolist())
print("Vectorized result:  ", is_hot_vectorized.tolist())
print("Results identical:  ", (is_hot_apply == is_hot_vectorized).all())
print("\nPrefer the vectorized form — it is faster.")

**Best practice:** Prefer vectorized operations (`> 25`, `* 2`, etc.) over `.apply(lambda ...)` whenever possible. `.apply()` is a last resort when no vectorized alternative exists.

## 7.3 String operations with `.str`

For `object` dtype Series (typically strings), Pandas provides a `.str` accessor that gives you vectorized string methods — no loops, no comprehensions.

In [ ]:
# Real-world: cleaning messy product name data
product_names = pd.Series(
    ["  MacBook Pro  ", "iphone 14", "AIRPODS MAX", "iPad mini", "  apple Watch"]
)

# Chain string operations — all vectorized
clean_names = (
    product_names
    .str.strip()   # Remove leading/trailing whitespace
    .str.title()   # Convert to title case
)

print("Original:")
print(product_names.tolist())
print("\nCleaned:")
print(clean_names.tolist())

In [ ]:
# More .str operations
emails = pd.Series(
    ["alice@gmail.com", "bob@yahoo.com", "carol@gmail.com", "dave@outlook.com"]
)

# Check if each email contains 'gmail'
is_gmail = emails.str.contains("gmail")
print("Is Gmail user:")
print(is_gmail)

# Extract the domain
domains = emails.str.split("@").str[1]  # Split on '@', take second part
print("\nDomains:")
print(domains)

---

# 8. Handling Missing Data in a Series

## What is missing data?

In the real world, data is almost never complete. A sensor goes offline. A form field is left blank. A join returns rows with no match. These are **missing values**.

Pandas represents missing values as `NaN` (Not a Number) for numeric data, or `None` / `pd.NA` for other types. NaN is a special float value defined by the IEEE 754 standard.

## Why this is tricky

`NaN` has unusual behavior. Any arithmetic with `NaN` returns `NaN`. And `NaN != NaN` is `True` (a quirk of the IEEE standard). So you **cannot check for NaN with `==`**.

In [ ]:
# NaN arithmetic quirks
print("np.nan + 5:      ", np.nan + 5)        # nan
print("np.nan * 0:      ", np.nan * 0)        # nan (not 0!)
print("np.nan == np.nan:", np.nan == np.nan)  # False — NaN is not equal to itself!

# Consequence: checking for NaN with == doesn't work
s = pd.Series([1.0, np.nan, 3.0])
print("\ns == np.nan (wrong approach):")
print(s == np.nan)   # All False

print("\npd.isna(s) (correct approach):")
print(pd.isna(s))    # Correct

## 8.1 Detecting missing values: `isna()` and `notna()`

`Series.isna()` and `Series.isnull()` are identical aliases. Both return a boolean Series: `True` where the value is `NaN` or `None`. They do **not** modify the original.

`Series.notna()` and `Series.notnull()` are the inverse.

In [ ]:
# Realistic sensor data with missing readings
temperature_readings = pd.Series(
    [22.1, None, 23.5, np.nan, 21.8, 24.0, None],
    index=pd.date_range("2024-01-01", periods=7, freq="D"),
    name="sensor_temp"
)

print("Readings:")
print(temperature_readings)

print("\nMissing mask (isna):")
print(temperature_readings.isna())

print("\nNumber of missing values:", temperature_readings.isna().sum())
print(f"Percentage missing:       {temperature_readings.isna().mean() * 100:.1f}%")

## 8.2 Dropping missing values: `dropna()`

Returns a **new** Series with all `NaN` rows removed. The original is not modified.

In [ ]:
clean_readings = temperature_readings.dropna()
print("After dropna():")
print(clean_readings)

print("\nOriginal still intact (dropna is non-mutating):")
print(temperature_readings.isna().sum(), "missing values remain in original")

## 8.3 Filling missing values: `fillna()`

## Syntax

```python
series.fillna(value=None, method=None, limit=None)
```

| Parameter | Purpose |
|---|---|
| `value` | A scalar, dict, or Series to fill with |
| `method` | `'ffill'` (forward fill) or `'bfill'` (backward fill) |
| `limit` | Max consecutive NaN rows to fill |

Returns a **new** Series (non-mutating).

In [ ]:
# Strategy 1: Fill with a constant (e.g., the mean)
mean_temp = temperature_readings.mean()
filled_with_mean = temperature_readings.fillna(mean_temp)
print("Filled with mean:")
print(filled_with_mean)
print()

In [ ]:
# Strategy 2: Forward fill — carry last known value forward
# Great for time series where the last reading is still valid
filled_forward = temperature_readings.ffill()
print("Forward fill (ffill):")
print(filled_forward)
print()

# Strategy 3: Backward fill
filled_backward = temperature_readings.bfill()
print("Backward fill (bfill):")
print(filled_backward)

## Best practice — choosing a fill strategy

| Data type | Recommended strategy |
|---|---|
| Time series (sensor, price) | `ffill()` — last known value |
| Symmetric data (heights, weights) | `fillna(mean)` |
| Skewed data | `fillna(median)` |
| Categorical data | `fillna(mode()[0])` — most frequent |
| When you do not know | `dropna()` and be honest about it |

**Common mistake:** Filling with the mean of the entire column without thinking. If the data has time dependency, forward fill is almost always more appropriate than mean fill.

---

# 9. Useful Series Methods

## 9.1 Descriptive statistics

In [ ]:
# Employee salary dataset
salaries = pd.Series(
    [52000, 68000, 75000, 48000, 91000, 63000, 55000, 82000, 70000, 45000],
    name="annual_salary"
)

# .describe() gives a fast statistical overview
print(salaries.describe())

In [ ]:
# Individual statistics
print(f"Mean:      {salaries.mean():,.0f}")
print(f"Median:    {salaries.median():,.0f}")
print(f"Std Dev:   {salaries.std():,.0f}")
print(f"Min:       {salaries.min():,.0f}")
print(f"Max:       {salaries.max():,.0f}")
print(f"Skewness:  {salaries.skew():.3f}")

**Note on NaN behavior:** Aggregation functions **skip NaN by default**. So `pd.Series([1, np.nan, 3]).mean()` returns `2.0`, not `nan`. Pass `skipna=False` if you want NaN to propagate.

## 9.2 Sorting

In [ ]:
# sort_values() — sort by the actual data values
print("Salaries sorted high to low:")
print(salaries.sort_values(ascending=False))

# sort_index() — sort by the index labels
named_salaries = pd.Series(
    {"Charlie": 75000, "Alice": 68000, "Bob": 91000, "Dave": 55000}
)
print("\nSorted alphabetically by name (sort_index):")
print(named_salaries.sort_index())

## 9.3 Value counts — for categorical data

In [ ]:
# Survey responses
survey_responses = pd.Series(
    ["satisfied", "neutral", "satisfied", "dissatisfied", "satisfied",
     "neutral", "satisfied", "dissatisfied", "neutral", "satisfied"],
    name="rating"
)

# Counts (sorted by frequency by default)
print("Value counts:")
print(survey_responses.value_counts())

# Proportions
print("\nAs proportions (normalize=True):")
print(survey_responses.value_counts(normalize=True).round(2))

## 9.4 Checking membership: `isin()`

In [ ]:
# Filter orders from specific states
order_states = pd.Series(
    ["Maharashtra", "Karnataka", "UP", "Tamil Nadu", "Maharashtra", "Goa", "Karnataka"]
)

# isin() is much cleaner than multiple OR conditions
south_states = ["Karnataka", "Tamil Nadu", "Goa"]
is_south = order_states.isin(south_states)

print("South India orders:")
print(order_states[is_south])

## 9.5 Cumulative operations and percentage change

In [ ]:
# Monthly revenue
monthly_revenue = pd.Series(
    [150_000, 175_000, 162_000, 198_000, 210_000, 225_000],
    index=["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    name="revenue"
)

print("Cumulative revenue:")
print(monthly_revenue.cumsum())

print("\nMonth-over-month growth (%):")
print((monthly_revenue.pct_change() * 100).round(1))
print("Note: Jan is NaN because there is no prior month")

## 9.6 Type conversion with `.astype()`

Returns a new Series with values cast to the specified dtype. Does **not** modify in place.

In [ ]:
# Prices loaded from CSV came in as strings — a very common situation
prices_as_strings = pd.Series(["99.99", "149.50", "9.00", "299.99"])
print("dtype before:", prices_as_strings.dtype)

prices_as_floats = prices_as_strings.astype(float)
print("dtype after: ", prices_as_floats.dtype)
print(prices_as_floats)

In [ ]:
# Common mistake: astype() on non-convertible data raises ValueError
bad_data = pd.Series(["100", "200", "three hundred", "400"])

try:
    bad_data.astype(float)
except ValueError as e:
    print(f"ValueError: {e}")

print("\nFix: use pd.to_numeric() with errors='coerce'")
print("This turns bad values into NaN instead of crashing:")
print(pd.to_numeric(bad_data, errors="coerce"))

**`pd.to_numeric()` with `errors='coerce'`** is one of the most useful tools for dirty data. It converts what it can and turns the rest into `NaN` without crashing your pipeline.

---

# 10. End-of-Notebook Summary, Cheat Sheet & Exercises

## Quick Revision

| Concept | Key point |
|---|---|
| Series | 1D labeled array. Values + Index. |
| Index | Labels for data. Not just row numbers. Enables alignment. |
| `.loc[]` | Label-based. Slices are **inclusive** on both ends. |
| `.iloc[]` | Position-based. Slices are **exclusive** on the right. |
| Vectorized ops | Use `+`, `-`, `*`, `>`, etc. directly. Never loop. |
| `.apply()` | Use when no vectorized alternative exists. Slower. |
| NaN | Use `isna()`, not `== np.nan`. Propagates through math. |
| `dropna()` | Removes NaN rows. Returns new Series. |
| `fillna()` | Replaces NaN. Returns new Series. |
| `dtype` | Single type per Series. `object` for strings (slow). |
| `astype()` | Converts type. Use `pd.to_numeric(errors='coerce')` for dirty data. |

## Key Takeaways

1. A Series is more than a list. It has a labeled index that enables alignment and protects against subtle positional bugs.
2. Always use `.loc` or `.iloc` explicitly. Avoid ambiguous `[]` access when your index contains integers.
3. Vectorized operations are the Pandas way. Avoid Python loops on Series.
4. `NaN` is not zero, not an empty string, and not comparable with `==`. Use `.isna()`.
5. `object` dtype is Pandas' slow path. Get your columns to numeric or category dtype early in your pipeline.

---

## 3 Interview Questions

**Q1:** What is the difference between `.loc` and `.iloc`? When would they return different results for the same integer argument?

**Q2:** Two Series have different index orders. You add them together. What does Pandas do? What happens to labels that only exist in one Series?

**Q3:** Why can you not check for NaN using `series == np.nan`? What is the correct approach?

---

## Practice Exercises

In [ ]:
# Exercise 1
# You have exam scores below.
# a) Find the average score, ignoring NaN.
# b) Replace NaN values with the median score.
# c) How many students scored above 70?

exam_scores = pd.Series(
    [85, 92, np.nan, 74, 68, np.nan, 91, 55, 78, 63],
    index=[f"student_{i+1}" for i in range(10)]
)

# Your solution here:


In [ ]:
# Exercise 2
# You have prices from two suppliers.
# a) Calculate the price difference for each product (supplier_a - supplier_b).
# b) Which products exist in only one supplier's catalog? (Hint: look for NaN in the difference)

supplier_a = pd.Series({"Laptop": 75000, "Phone": 28000, "Tablet": 35000, "Watch": 18000})
supplier_b = pd.Series({"Phone": 27500, "Tablet": 36000, "Earbuds": 4500, "Watch": 17200})

# Your solution here:


In [ ]:
# Exercise 3
# Clean the following messy salary data:
# a) Convert to numeric (coerce bad values to NaN)
# b) Remove salaries that are 0, negative, or NaN
# c) Report mean, median, and count of valid salaries

raw_salaries = pd.Series(
    ["52000", "68000", "N/A", "75000", "-1000", "0", "91000", "bad_value", "63000"]
)

# Your solution here:


---

# Practice Section — Full Coverage

These exercises cover every topic in this notebook.
Work through them in order — difficulty increases gradually.

**Difficulty guide:**

- `[E]` Easy — one concept, direct application
- `[M]` Medium — two or more concepts combined
- `[H]` Hard — real-world thinking required

Do not look at the hint until you have tried for at least 5 minutes.

## Level 1 — Easy `[E]`

### Q1 `[E]` — Creating a Series

Create a Series called `monthly_expenses` with the following data:

| Month | Expense (INR) |
|---|---|
| Jan | 15200 |
| Feb | 18400 |
| Mar | 12100 |
| Apr | 21000 |
| May | 16500 |

Tasks:
- Print the Series
- Print the dtype
- Print the index
- Print the total expenses using a single method

In [ ]:
import pandas as pd
import numpy as np

# Q1 — Your solution here


<details>
<summary>Hint (click to expand)</summary>

Create from a dict. Use `.sum()` for total.

</details>

### Q2 `[E]` — loc vs iloc

Given the Series below, answer each question using **only `.loc` or `.iloc`** (not plain `[]`).

- a) Get the score of `'Riya'` using `.loc`
- b) Get the **third** student's score using `.iloc`
- c) Get scores of `'Arjun'` and `'Meera'` using `.loc`
- d) Get the **last two** scores using `.iloc`

In [ ]:
student_scores = pd.Series(
    [88, 74, 91, 63, 79, 85],
    index=["Arjun", "Priya", "Riya", "Karan", "Meera", "Dev"]
)

# Q2a — Riya's score using .loc

# Q2b — Third student's score using .iloc

# Q2c — Arjun and Meera using .loc

# Q2d — Last two scores using .iloc


### Q3 `[E]` — Vectorized operations

You have product prices in USD. Do the following **without any loops**:

- a) Convert all prices to INR (use rate = 83.5)
- b) Apply a 10% discount to all prices (in USD)
- c) Find which products cost more than $50 USD
- d) Round the INR prices to nearest integer

In [ ]:
product_prices_usd = pd.Series(
    {"Keyboard": 45.99, "Mouse": 29.99, "Monitor": 189.99, "Webcam": 59.99, "Headset": 79.99}
)

# Q3a — Convert to INR

# Q3b — Apply 10% discount

# Q3c — Products above $50

# Q3d — Round INR prices


### Q4 `[E]` — Basic missing data

Given the rainfall data below:

- a) How many months have missing data?
- b) What percentage of data is missing?
- c) Drop all missing months
- d) Fill missing values with the **median** rainfall

In [ ]:
rainfall_mm = pd.Series(
    [12.5, None, 8.3, 45.1, None, None, 22.0, 18.7, None, 5.2, 31.4, 9.8],
    index=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    name="rainfall_mm"
)

# Q4a — Count missing

# Q4b — Percentage missing

# Q4c — Drop missing

# Q4d — Fill with median


### Q5 `[E]` — String operations

Clean this employee name data:

- a) Strip whitespace from all names
- b) Convert to proper title case
- c) Find all employees whose name **starts with 'A'**
- d) Count how many characters each cleaned name has

In [ ]:
employee_names = pd.Series(
    ["  alice sharma ", "BOB MEHTA", "carol SINGH", "  ankit joshi", "DIVYA patel  ", "arun kumar"]
)

# Q5a — Strip whitespace

# Q5b — Title case

# Q5c — Names starting with 'A' (case-insensitive)

# Q5d — Character count per name


---

## Level 2 — Medium `[M]`

### Q6 `[M]` — Index alignment trap

Two teams tracked runs scored in cricket matches.
The matches are labeled by match ID.

- a) Add the scores of both teams for each match (total runs)
- b) Which matches have data for **both** teams? (no NaN in result)
- c) Which matches have data for **only one** team?
- d) For missing values in the sum, fill with the score of whichever team did play

*Hint for (d): use `fillna()` after adding.*

In [ ]:
team_a_runs = pd.Series({"M01": 187, "M02": 203, "M03": 156, "M05": 221})
team_b_runs = pd.Series({"M01": 175, "M03": 198, "M04": 211, "M05": 189})

# Q6a — Total runs per match

# Q6b — Matches with both teams

# Q6c — Matches with only one team

# Q6d — Fill NaN with available team's score


<details>
<summary>Hint for Q6d</summary>

After adding `team_a_runs + team_b_runs`, NaN appears where one team is missing.
For those NaN matches, use `fillna()` with `team_a_runs.add(team_b_runs, fill_value=0)`.

</details>

### Q7 `[M]` — Filtering with multiple conditions

Using the job listings below:

- a) Find all jobs with salary **between 60,000 and 100,000** (inclusive)
- b) Find all jobs that are **NOT** in `['HR', 'Support']` departments
- c) Find jobs where salary is **above the mean** salary
- d) What is the **rank** of each job by salary (highest = rank 1)?

In [ ]:
job_salaries = pd.Series(
    {"Engineer": 95000, "Manager": 120000, "Analyst": 72000,
     "HR": 55000, "Designer": 80000, "Support": 42000,
     "Lead": 110000, "Intern": 25000}
)

# Q7a — Between 60,000 and 100,000

# Q7b — Not HR or Support

# Q7c — Above mean salary

# Q7d — Rank by salary (hint: look up .rank() method)


### Q8 `[M]` — apply() vs vectorization

Given a Series of transaction amounts:

- a) Classify each transaction: `'low'` (< 1000), `'medium'` (1000–9999), `'high'` (>= 10000). Use `.apply()`.
- b) Now do the same classification using `pd.cut()` instead of `.apply()`. Compare.
- c) Add a 5% tax to amounts above 5000, and 0% below. Use `.apply()`.
- d) Now do (c) using `np.where()` — which is faster?

*`pd.cut()` and `np.where()` are new — look them up, figure out the syntax.*

In [ ]:
transactions = pd.Series(
    [500, 12000, 3400, 750, 8900, 25000, 1200, 450, 15000, 6700]
)

# Q8a — Classify using apply()

# Q8b — Classify using pd.cut()

# Q8c — Tax using apply()

# Q8d — Tax using np.where()


### Q9 `[M]` — Data type cleanup pipeline

This is a realistic scenario. You received a CSV column with messy data.
Build a cleaning pipeline (step by step) to fix it:

- a) Convert to numeric — bad values become NaN
- b) Replace negative values with NaN (they are data entry errors)
- c) Fill NaN with the **mean** of valid values
- d) Convert the final result to `int` dtype
- e) Print: original count, valid count after step (b), and final dtype

In [ ]:
raw_age_data = pd.Series(
    ["25", "32", "N/A", "28", "-5", "41", "unknown", "36", "-1", "29", "0", "33"]
)

# Q9a — Convert to numeric

# Q9b — Replace negatives with NaN

# Q9c — Fill NaN with mean of valid values

# Q9d — Convert to int

# Q9e — Print summary


<details>
<summary>Hint for Q9b</summary>

After converting to numeric, use boolean indexing to set negatives to NaN:
```python
s[s < 0] = np.nan
```
Or use `np.where()` or `.mask()`.

</details>

---

## Level 3 — Hard `[H]`

### Q10 `[H]` — Full analysis pipeline

You have daily stock closing prices for a month.
Perform a complete analysis:

- a) Calculate the **daily return** (% change from previous day)
- b) Find the **best day** and **worst day** (by daily return)
- c) How many days had a **positive return** vs **negative return**?
- d) Calculate a **5-day rolling average** of the price (use `.rolling(5).mean()`)
- e) Identify days where the price was **above the rolling average** (potential trend signal)
- f) What was the **total return** from day 1 to the last day (in %)?

In [ ]:
import numpy as np

np.random.seed(42)  # For reproducibility

# Simulated stock prices for 22 trading days
closing_prices = pd.Series(
    [2845, 2891, 2873, 2920, 2905, 2938, 2967, 2954, 2989, 3012,
     2998, 3021, 3045, 3038, 3067, 3089, 3074, 3102, 3091, 3118,
     3135, 3127],
    index=pd.bdate_range(start="2024-01-01", periods=22),  # business days
    name="NIFTY50_close"
)

# Q10a — Daily return (%)

# Q10b — Best and worst day

# Q10c — Positive vs negative days

# Q10d — 5-day rolling average

# Q10e — Days above rolling average

# Q10f — Total return from day 1 to last day


### Q11 `[H]` — Real-world: election data

You have vote counts for parties across 6 constituencies.

- a) Find the **winner** of each constituency (party with most votes)
  - *Hint: this is not straightforward with a flat Series — think about it*
- b) What is the **total vote share** of each party across all constituencies (in %)?
- c) Which party won the **most constituencies**?
- d) Identify constituencies where the **winning margin** was less than 5,000 votes (close contests)
- e) Which party had the **highest average** votes per constituency?

In [ ]:
# Vote data: each row = (constituency, party)
# We'll use a flat structure — part of the challenge is working with it

votes_raw = {
    "C1_PartyA": 48200, "C1_PartyB": 39100, "C1_PartyC": 12500,
    "C2_PartyA": 31000, "C2_PartyB": 52400, "C2_PartyC": 18900,
    "C3_PartyA": 44500, "C3_PartyB": 41200, "C3_PartyC": 9800,
    "C4_PartyA": 27300, "C4_PartyB": 29800, "C4_PartyC": 38100,
    "C5_PartyA": 55600, "C5_PartyB": 51200, "C5_PartyC": 14300,
    "C6_PartyA": 33100, "C6_PartyB": 38400, "C6_PartyC": 41700,
}

votes = pd.Series(votes_raw)
print(votes)
print()

# Hint: you can extract constituency and party using .str operations
# votes.index.str.split('_').str[0]  → constituency
# votes.index.str.split('_').str[1]  → party

# Q11a — Winner of each constituency

# Q11b — Total vote share per party

# Q11c — Party that won most constituencies

# Q11d — Close contests (winning margin < 5000)

# Q11e — Highest average votes per constituency per party


<details>
<summary>Hint for Q11</summary>

For (a): group by constituency using `groupby`, then find the index of max value per group.
But since we only have a Series (not a DataFrame), you will need to use:
- `votes.index.str.split('_').str[0]` to get constituencies
- `votes.groupby(constituencies).idxmax()` to get the label of max vote count per constituency
- Then extract the party from that label using `.str.split('_').str[1]`

This is genuinely hard — it is a preview of GroupBy (Notebook 5).

</details>

---

## Mini Assignment — End-to-End Series Project

**Scenario:** You are a data analyst at an e-commerce company. You receive a raw export of order values for the past 30 days. The data has quality issues.

**Your job:** Build a complete data cleaning and reporting pipeline using only Series operations learned in this notebook.

**Deliverables (each as a print statement):**

1. How many orders are in the raw data? How many are invalid (NaN or negative)?
2. After cleaning, what is the mean, median, and standard deviation of order value?
3. Classify orders into tiers: `'small'` (< 500), `'medium'` (500–1999), `'large'` (>= 2000). How many in each tier?
4. What percentage of orders are `'large'`?
5. What is the 7-day rolling average of daily order value? On which day was the rolling average highest?
6. The company wants to flag the **top 10% of orders by value** for review. Which days have these high-value orders?

*Note: The data has one order per day. Day 1 is January 1, 2024.*

In [ ]:
import pandas as pd
import numpy as np

# Raw order value data — contains errors intentionally
raw_order_values = pd.Series(
    [1200, 450, -300, 3400, 890, None, 2100, 560, 1750, 320,
     4500, None, 280, 1900, 670, -50, 3200, 1100, 820, None,
     2800, 490, 1600, 3700, 950, 1400, None, 2300, 780, 4100],
    index=pd.date_range(start="2024-01-01", periods=30, freq="D"),
    name="order_value_inr"
)

# ── Step 1: Understand the raw data ──────────────────────────────


# ── Step 2: Clean the data ───────────────────────────────────────
# (remove NaN and negative values, fill appropriately)


# ── Step 3: Compute statistics ───────────────────────────────────


# ── Step 4: Classify into tiers ──────────────────────────────────


# ── Step 5: Rolling average ──────────────────────────────────────


# ── Step 6: Flag top 10% ─────────────────────────────────────────
# Hint: use .quantile(0.90) to find the 90th percentile threshold



---

## Answer Key — Quick Reference

Refer to these **only after** you have attempted each question.

Seeing the answer first defeats the purpose. Struggle is how you build intuition.

In [ ]:
# ── Q1 Answer ────────────────────────────────────────────────────────────────
monthly_expenses = pd.Series(
    {"Jan": 15200, "Feb": 18400, "Mar": 12100, "Apr": 21000, "May": 16500}
)
print(monthly_expenses)
print("dtype:", monthly_expenses.dtype)
print("index:", monthly_expenses.index.tolist())
print("total:", monthly_expenses.sum())

In [ ]:
# ── Q2 Answer ────────────────────────────────────────────────────────────────
student_scores = pd.Series(
    [88, 74, 91, 63, 79, 85],
    index=["Arjun", "Priya", "Riya", "Karan", "Meera", "Dev"]
)
print("a) Riya:",        student_scores.loc["Riya"])
print("b) Third:",       student_scores.iloc[2])
print("c) Arjun & Meera:", student_scores.loc[["Arjun", "Meera"]].tolist())
print("d) Last two:",    student_scores.iloc[-2:].tolist())

In [ ]:
# ── Q3 Answer ────────────────────────────────────────────────────────────────
product_prices_usd = pd.Series(
    {"Keyboard": 45.99, "Mouse": 29.99, "Monitor": 189.99, "Webcam": 59.99, "Headset": 79.99}
)
print("a) INR prices:\n",    (product_prices_usd * 83.5).round(0))
print("b) After 10% off:\n", (product_prices_usd * 0.90).round(2))
print("c) Above $50:\n",     product_prices_usd[product_prices_usd > 50])
print("d) INR rounded:\n",   (product_prices_usd * 83.5).round(0).astype(int))

In [ ]:
# ── Q4 Answer ────────────────────────────────────────────────────────────────
rainfall_mm = pd.Series(
    [12.5, None, 8.3, 45.1, None, None, 22.0, 18.7, None, 5.2, 31.4, 9.8],
    index=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
)
print("a) Missing count:",      rainfall_mm.isna().sum())
print("b) Missing %:",          f"{rainfall_mm.isna().mean()*100:.1f}%")
print("c) After dropna:\n",     rainfall_mm.dropna())
print("d) Filled w/ median:\n", rainfall_mm.fillna(rainfall_mm.median()))

In [ ]:
# ── Q5 Answer ────────────────────────────────────────────────────────────────
employee_names = pd.Series(
    ["  alice sharma ", "BOB MEHTA", "carol SINGH", "  ankit joshi", "DIVYA patel  ", "arun kumar"]
)
cleaned = employee_names.str.strip().str.title()
print("a+b) Cleaned:\n", cleaned.tolist())
starts_with_a = cleaned[cleaned.str.startswith("A")]
print("c) Starts with A:\n", starts_with_a.tolist())
print("d) Name lengths:\n", cleaned.str.len().tolist())

In [ ]:
# ── Q6 Answer ────────────────────────────────────────────────────────────────
team_a_runs = pd.Series({"M01": 187, "M02": 203, "M03": 156, "M05": 221})
team_b_runs = pd.Series({"M01": 175, "M03": 198, "M04": 211, "M05": 189})

total = team_a_runs + team_b_runs
print("a) Total runs (NaN = only one team played):\n", total)

both = total[total.notna()]
print("\nb) Matches with both teams:", both.index.tolist())

only_one = total[total.isna()]
print("c) Matches with only one team:", only_one.index.tolist())

# For NaN matches, at least one team's score is available
filled_total = team_a_runs.add(team_b_runs, fill_value=0)
print("\nd) Filled total (using fill_value=0 for missing team):\n", filled_total)

In [ ]:
# ── Q7 Answer ────────────────────────────────────────────────────────────────
job_salaries = pd.Series(
    {"Engineer": 95000, "Manager": 120000, "Analyst": 72000,
     "HR": 55000, "Designer": 80000, "Support": 42000,
     "Lead": 110000, "Intern": 25000}
)
print("a) Between 60K-100K:\n",    job_salaries[(job_salaries >= 60000) & (job_salaries <= 100000)])
print("b) Not HR or Support:\n",   job_salaries[~job_salaries.index.isin(["HR", "Support"])])
print("c) Above mean salary:\n",   job_salaries[job_salaries > job_salaries.mean()])
print("d) Salary rank:\n",         job_salaries.rank(ascending=False).astype(int))

In [ ]:
# ── Q8 Answer ────────────────────────────────────────────────────────────────
transactions = pd.Series([500, 12000, 3400, 750, 8900, 25000, 1200, 450, 15000, 6700])

# a) apply()
def classify(x):
    if x < 1000: return "low"
    elif x < 10000: return "medium"
    return "high"

print("a) via apply():\n",  transactions.apply(classify).tolist())

# b) pd.cut()
tier_cut = pd.cut(transactions, bins=[0, 999, 9999, float("inf")], labels=["low","medium","high"])
print("b) via pd.cut():\n", tier_cut.tolist())

# c) Tax using apply()
taxed_apply = transactions.apply(lambda x: x * 1.05 if x > 5000 else x)
print("c) Tax via apply():\n",   taxed_apply.tolist())

# d) Tax using np.where() — vectorized, faster
taxed_where = pd.Series(np.where(transactions > 5000, transactions * 1.05, transactions))
print("d) Tax via np.where():\n", taxed_where.tolist())
print("Both identical:", (taxed_apply.values == taxed_where.values).all())

In [ ]:
# ── Q9 Answer ────────────────────────────────────────────────────────────────
raw_age_data = pd.Series(
    ["25", "32", "N/A", "28", "-5", "41", "unknown", "36", "-1", "29", "0", "33"]
)

# a) Convert to numeric
ages = pd.to_numeric(raw_age_data, errors="coerce")

# b) Replace negative and zero values with NaN
ages = ages.where(ages > 0, other=np.nan)  # keep only positive values
valid_count = ages.notna().sum()

# c) Fill NaN with mean of valid values
ages_filled = ages.fillna(ages.mean())

# d) Convert to int
ages_final = ages_filled.round(0).astype(int)

# e) Summary
print(f"e) Original count:  {len(raw_age_data)}")
print(f"   Valid after (b):  {valid_count}")
print(f"   Final dtype:      {ages_final.dtype}")
print("\nFinal cleaned ages:")
print(ages_final.tolist())

---

## Series Cheat Sheet

```python
import pandas as pd
import numpy as np

# ── Creation ──────────────────────────────────────────────────
s = pd.Series([1, 2, 3], index=['a','b','c'], name='values')
s = pd.Series({'a': 1, 'b': 2})               # from dict
s = pd.Series(0, index=['a','b','c'])          # scalar broadcast

# ── Inspection ────────────────────────────────────────────────
s.dtype           # data type
s.shape           # (n,)
s.index           # index labels
s.values          # NumPy array
s.name            # Series name
s.describe()      # summary stats

# ── Selection ─────────────────────────────────────────────────
s.loc['a']        # label-based, single (inclusive slice)
s.loc['a':'c']    # label-based slice — INCLUSIVE on both ends
s.iloc[0]         # position-based, single
s.iloc[0:2]       # position-based slice — EXCLUSIVE on right
s[s > 1]          # boolean indexing
s.isin(['a','b']) # membership test

# ── Missing values ────────────────────────────────────────────
s.isna()          # True where NaN
s.notna()         # True where not NaN
s.dropna()        # remove NaN rows (new Series)
s.fillna(0)       # fill NaN with scalar (new Series)
s.ffill()         # forward fill
s.bfill()         # backward fill

# ── Operations ────────────────────────────────────────────────
s * 2             # vectorized multiply
s.apply(func)     # element-wise function (slow — last resort)
s.str.strip()     # string accessor
s.astype(float)   # type conversion
pd.to_numeric(s, errors='coerce')   # safe conversion

# ── Statistics ────────────────────────────────────────────────
s.mean(), s.median(), s.std()
s.min(), s.max(), s.sum()
s.value_counts()  # frequency table
s.sort_values()   # sort by value
s.sort_index()    # sort by label
s.cumsum()        # cumulative sum
s.pct_change()    # % change between elements
```

---

## What's Next?

**Notebook 2** will cover the **DataFrame** in the same depth:
- Creating DataFrames from multiple sources
- Column selection, row selection, and method chaining
- Adding, removing, and renaming columns
- Filtering rows with complex conditions
- The `.query()` method
- The difference between views and copies (`SettingWithCopyWarning`)

Master the Series first. The DataFrame is just a collection of Series sharing an index — and everything you learned here transfers directly.